# FEATURES CREATION
## CREATIONS DE QUELQUES FEATURES EN PLUS POUR LE MODELE DE ML 

In [289]:
import pandas as pd

path = "../data/"
ufc = pd.read_csv(path+"UFC.csv")
fighters = pd.read_csv(path+"fighter_details.csv")
fight_stats = pd.read_csv(path+"fight_details.csv")


In [290]:
# On garde uniquement ce qui est utile
cols = ["fight_id", "date", "r_name", "b_name", "winner", "method"]

df = ufc[cols].copy()
df["date"] = pd.to_datetime(df["date"])

# Format long
red = df[["fight_id", "date", "r_name", "winner", "method"]].copy()
red.columns = ["fight_id", "date", "fighter", "winner", "method"]
red["corner"] = "red"

blue = df[["fight_id", "date", "b_name", "winner", "method"]].copy()
blue.columns = ["fight_id", "date", "fighter", "winner", "method"]
blue["corner"] = "blue"

long = pd.concat([red, blue], ignore_index=True)

# Résultat du fighter dans ce combat
long["win"] = (long["fighter"] == long["winner"]).astype(int)



In [291]:
long = long.sort_values(["fighter", "date"])


In [292]:
long["n_fights_before"] = long.groupby("fighter").cumcount()


In [293]:
long["is_ko"] = ((long["win"] == 1) & (long["method"].str.contains("KO", na=False))).astype(int)
long["is_sub"] = ((long["win"] == 1) & (long["method"].str.contains("SUB", na=False))).astype(int)

long["ko_before"] = long.groupby("fighter")["is_ko"].cumsum().shift(1)
long["sub_before"] = long.groupby("fighter")["is_sub"].cumsum().shift(1)

long[["ko_before", "sub_before"]] = long[["ko_before", "sub_before"]].fillna(0)


In [294]:
# Assurer l'ordre temporel
long = long.sort_values(["fighter", "date"])

# Date du combat précédent
long["prev_fight_date"] = long.groupby("fighter")["date"].shift(1)

# Temps depuis le dernier combat (en jours)
long["days_since_last_fight"] = (
    long["date"] - long["prev_fight_date"]
).dt.days

# NA = premier combat → on laisse NA ou on remplira plus tard
long["days_since_last_fight"].describe()


count    14064.000000
mean       218.468003
std        215.227669
min          0.000000
25%        126.000000
50%        168.000000
75%        244.000000
max       4180.000000
Name: days_since_last_fight, dtype: float64

# PLUS LONGUE SERIE DE VICTOIRE

In [295]:
def longest_win_streak(series):
    streak = max_streak = 0
    for x in series:
        if x == 1:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak

long["max_win_streak_before"] = (
    long.groupby("fighter")["win"]
        .apply(lambda x: x.shift(1).expanding().apply(longest_win_streak))
        .reset_index(level=0, drop=True)
)


# NOMBRE DE FIGHTS DURANT LES DERNIERS 365 JOURS

In [296]:
# Fonction rolling par fighter
def fights_last_year(dates):
    counts = []
    for i, d in enumerate(dates):
        counts.append(
            ((dates < d) & (dates >= d - pd.Timedelta(days=365))).sum()
        )
    return counts

long["fights_last_1y"] = (
    long.groupby("fighter")["date"]
        .transform(fights_last_year)
)

long[[
    "fighter",
    "date",
    "days_since_last_fight",
    "fights_last_1y"
]].head(10)


,fighter,date,days_since_last_fight,fights_last_1y
9141,AJ Cunningham,2024-03-02,NaN,0
8589,AJ Cunningham,2025-03-15,378.0,0
1875,AJ Dobson,2022-02-12,NaN,0
9810,AJ Dobson,2022-10-22,252.0,1
1047,AJ Dobson,2023-08-12,294.0,1
9095,AJ Dobson,2024-03-23,224.0,1
10155,AJ Fletcher,2022-03-12,NaN,0
1566,AJ Fletcher,2022-08-20,161.0,1
1348,AJ Fletcher,2023-02-18,182.0,2
9327,AJ Fletcher,2023-09-23,217.0,1


# NOMBRE DE KO, DE SUB, DE VICTOIRE AU MOMENT D UN COMBAT AU TEMPS T 

In [297]:
# Sécurisation
long = long.reset_index(drop=True)

features = long[
    [
        "fight_id",
        "fighter",
        "n_fights_before",
        "ko_before",
        "sub_before",
        "max_win_streak_before",
        "days_since_last_fight",
        "fights_last_1y"
    ]
].copy()


red_feat = (
    features
    .merge(
        ufc[["fight_id", "r_name"]],
        left_on=["fight_id", "fighter"],
        right_on=["fight_id", "r_name"],
        how="inner"
    )
    .drop(columns=["fighter", "r_name"])
    .rename(columns={
        "n_fights_before": "r_n_fights_before",
        "ko_before": "r_ko_before",
        "sub_before": "r_sub_before",
        "max_win_streak_before": "r_max_win_streak_before",
        "days_since_last_fight": "r_days_since_last_fight",
        "fights_last_1y": "r_fights_last_1y"
    })
)

blue_feat = (
    features
    .merge(
        ufc[["fight_id", "b_name"]],
        left_on=["fight_id", "fighter"],
        right_on=["fight_id", "b_name"],
        how="inner"
    )
    .drop(columns=["fighter", "b_name"])
    .rename(columns={
        "n_fights_before": "b_n_fights_before",
        "ko_before": "b_ko_before",
        "sub_before": "b_sub_before",
        "max_win_streak_before": "b_max_win_streak_before",
        "days_since_last_fight": "b_days_since_last_fight",
        "fights_last_1y": "b_fights_last_1y"
    })
)


df_features = (
    ufc
    .merge(red_feat, on="fight_id", how="left")
    .merge(blue_feat, on="fight_id", how="left")
)


# Vérifier que les features existent
df_features[[
    "r_n_fights_before", "b_n_fights_before",
    "r_ko_before", "b_ko_before",
    "r_sub_before", "b_sub_before",
    "r_max_win_streak_before", "b_max_win_streak_before",
    "r_days_since_last_fight", "b_days_since_last_fight",
    "r_fights_last_1y", "b_fights_last_1y"
]].describe()


df_features.head().to_csv('check.csv')


In [298]:
# 1. Variables in-fight / post-fight (LEAKAGE)
infight_cols = [
    # Red
    "r_kd", "r_sig_str_landed", "r_sig_str_atmpted", "r_sig_str_acc",
    "r_total_str_landed", "r_total_str_atmpted", "r_total_str_acc",
    "r_td_landed", "r_td_atmpted", "r_td_acc",
    "r_sub_att", "r_ctrl",
    "r_head_landed", "r_head_atmpted", "r_head_acc",
    "r_body_landed", "r_body_atmpted", "r_body_acc",
    "r_leg_landed", "r_leg_atmpted", "r_leg_acc",
    "r_dist_landed", "r_dist_atmpted", "r_dist_acc",
    "r_clinch_landed", "r_clinch_atmpted", "r_clinch_acc",
    "r_ground_landed", "r_ground_atmpted", "r_ground_acc",
    "r_landed_head_per", "r_landed_body_per", "r_landed_leg_per",
    "r_landed_dist_per", "r_landed_clinch_per", "r_landed_ground_per",

    # Blue
    "b_kd", "b_sig_str_landed", "b_sig_str_atmpted", "b_sig_str_acc",
    "b_total_str_landed", "b_total_str_atmpted", "b_total_str_acc",
    "b_td_landed", "b_td_atmpted", "b_td_acc",
    "b_sub_att", "b_ctrl",
    "b_head_landed", "b_head_atmpted", "b_head_acc",
    "b_body_landed", "b_body_atmpted", "b_body_acc",
    "b_leg_landed", "b_leg_atmpted", "b_leg_acc",
    "b_dist_landed", "b_dist_atmpted", "b_dist_acc",
    "b_clinch_landed", "b_clinch_atmpted", "b_clinch_acc",
    "b_ground_landed", "b_ground_atmpted", "b_ground_acc",
    "b_landed_head_per", "b_landed_body_per", "b_landed_leg_per",
    "b_landed_dist_per", "b_landed_clinch_per", "b_landed_ground_per",
]


# 2. Labels (à garder séparément)
label_cols = [
    "winner", "method", "finish_round", "match_time_sec", "winner_id"
]


# 3. Identifiants / texte non prédictifs
meta_drop_cols = [
    "event_id", "event_name", "location", "referee",
    "r_id", "b_id", "r_nick_name", "b_nick_name"
]



In [299]:
cols_to_drop = infight_cols + label_cols + meta_drop_cols

df_clean = df_features.drop(
    columns=[c for c in cols_to_drop if c in df_features.columns],
    errors="ignore"
)


In [300]:
print("Nombre de colonnes AVANT :", df_features.shape[1])
print("Nombre de colonnes APRÈS :", df_clean.shape[1])

print(df_clean.columns.tolist())
df_clean.to_csv('dataset_final.csv')
print(df_clean.columns)

Nombre de colonnes AVANT : 136
Nombre de colonnes APRÈS : 51
['date', 'fight_id', 'division', 'title_fight', 'total_rounds', 'r_name', 'r_wins', 'r_losses', 'r_draws', 'r_height', 'r_weight', 'r_reach', 'r_stance', 'r_dob', 'r_splm', 'r_str_acc', 'r_sapm', 'r_str_def', 'r_td_avg', 'r_td_avg_acc', 'r_td_def', 'r_sub_avg', 'b_name', 'b_wins', 'b_losses', 'b_draws', 'b_height', 'b_weight', 'b_reach', 'b_stance', 'b_dob', 'b_splm', 'b_str_acc', 'b_sapm', 'b_str_def', 'b_td_avg', 'b_td_avg_acc', 'b_td_def', 'b_sub_avg', 'r_n_fights_before', 'r_ko_before', 'r_sub_before', 'r_max_win_streak_before', 'r_days_since_last_fight', 'r_fights_last_1y', 'b_n_fights_before', 'b_ko_before', 'b_sub_before', 'b_max_win_streak_before', 'b_days_since_last_fight', 'b_fights_last_1y']
Index(['date', 'fight_id', 'division', 'title_fight', 'total_rounds', 'r_name',
       'r_wins', 'r_losses', 'r_draws', 'r_height', 'r_weight', 'r_reach',
       'r_stance', 'r_dob', 'r_splm', 'r_str_acc', 'r_sapm', 'r_str_def'

In [301]:
# y = 1 si Red gagne, 0 sinon
y_win = (df_features["winner"] == df_features["r_name"]).astype(int)
y_win.value_counts(normalize=True)


1    0.634641
0    0.365359
Name: proportion, dtype: float64

# CALCULER L'AGE DU COMBATTANT AU MOMENT DU COMBAT

In [302]:
import pandas as pd

# Conversion en datetime
df_clean['r_dob'] = pd.to_datetime(df_clean['r_dob'], errors='coerce')
df_clean['b_dob'] = pd.to_datetime(df_clean['b_dob'], errors='coerce')
df_clean['date']  = pd.to_datetime(df_clean['date'], errors='coerce')

# Âges
df_clean['r_age'] = ((df_clean['date'] - df_clean['r_dob']).dt.days / 365.25).round(2)
df_clean['b_age'] = ((df_clean['date'] - df_clean['b_dob']).dt.days / 365.25).round(2)

df_clean = df_clean[
    (df_clean['r_age'].between(16, 60)) &
    (df_clean['b_age'].between(16, 60))
]



# DIVISION CLEANING (TROP DE DIVISIONS INUTILES ON GARDE LES 11 CLASSIQUES)

In [324]:
# ============================
# NETTOYAGE DES DIVISIONS UFC
# ============================

print(df_clean.shape)
# 1. Normalisation robuste des noms
df_clean["division"] = (
    df_clean["division"]
    .str.lower()
    .str.strip()
    .str.replace("’", "'", regex=False)
)

# 2. Divisions UFC STANDARD à conserver
valid_divisions = [
    "lightweight",
    "welterweight",
    "middleweight",
    "featherweight",
    "bantamweight",
    "light heavyweight",
    "heavyweight",
    "flyweight",
    "women's strawweight",
    "women's flyweight",
    "women's bantamweight",
]

# 3. Patterns à EXCLURE (règle générale)
exclude_patterns = [
    "interim",
    "tournament",
    "ultimate fighter",
    "tuf",
    "road to",
    "superfight",
    "super heavyweight",
    "open weight",
    "catch weight",
    "ultimate ultimate",
]

# 4. Création du masque d'exclusion
mask_exclude = df_clean["division"].apply(
    lambda x: any(pat in x for pat in exclude_patterns)
)

# 5. Filtrage final
df_clean = df_clean[
    df_clean["division"].isin(valid_divisions) & (~mask_exclude)
].reset_index(drop=True)

# 6. Vérification finale
print("Divisions conservées :")
print(sorted(df_clean["division"].unique()))

print("\nRépartition (%) :")
print(
    df_clean["division"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


print(df_clean.columns)




(7907, 53)
Divisions conservées :
['bantamweight', 'featherweight', 'flyweight', 'heavyweight', 'light heavyweight', 'lightweight', 'middleweight', 'welterweight', "women's bantamweight", "women's flyweight", "women's strawweight"]

Répartition (%) :
division
lightweight             17.15
welterweight            16.78
middleweight            13.42
featherweight           10.21
bantamweight             9.22
light heavyweight        9.04
heavyweight              8.80
flyweight                4.84
women's strawweight      4.36
women's flyweight        3.29
women's bantamweight     2.88
Name: proportion, dtype: float64
Index(['date', 'fight_id', 'division', 'title_fight', 'total_rounds', 'r_name',
       'r_wins', 'r_losses', 'r_draws', 'r_height', 'r_weight', 'r_reach',
       'r_stance', 'r_dob', 'r_splm', 'r_str_acc', 'r_sapm', 'r_str_def',
       'r_td_avg', 'r_td_avg_acc', 'r_td_def', 'r_sub_avg', 'b_name', 'b_wins',
       'b_losses', 'b_draws', 'b_height', 'b_weight', 'b_reach', 'b_

# SUPPRIMER VARIABLES CATEGORIELLES PLUS LANCEMENT D'UN PREMIER MODELE AVEC COIN RED/BLUE NON RANDOMISÉ

In [334]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

# ============================
# 1. CONSTRUCTION DE y (ALIGNÉ)
# ============================

y = (
    df_features.loc[df_clean.index, "winner"]
    == df_features.loc[df_clean.index, "r_name"]
).astype(int)

# ============================
# 2. CONSTRUCTION DE X
# ============================

cols_to_drop = [
    "date",
    "fight_id",
    "r_name",
    "b_name",
    "title_fight",
    "division",
    "r_stance",
    "b_stance",
]

X = df_clean.drop(
    columns=[c for c in cols_to_drop if c in df_clean.columns]
)

# Supprimer toutes les colonnes datetime
X = X.drop(columns=X.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns)

# Sécurités
assert X.select_dtypes(include="object").shape[1] == 0
assert X.select_dtypes(include=["datetime64[ns]", "datetime64"]).shape[1] == 0

# ============================
# 3. SUPPRESSION DES NaN (ALIGNÉE)
# ============================

valid_idx = X.dropna().index

X = X.loc[valid_idx]
y = y.loc[valid_idx]
df_clean = df_clean.loc[valid_idx]

# Sécurité
assert X.isna().sum().sum() == 0

# ============================
# 4. SPLIT TEMPOREL
# ============================

dates = pd.to_datetime(df_clean["date"])
split_date = "2025-01-01"

train_idx = dates < split_date
test_idx  = dates >= split_date

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print("Train size :", X_train.shape[0])
print("Test size  :", X_test.shape[0])

# ============================
# 5. SCALING
# ============================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ============================
# 6. LOGISTIC REGRESSION
# ============================

log_reg = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    n_jobs=-1
)

log_reg.fit(X_train_scaled, y_train)

# ============================
# 7. PRÉDICTIONS
# ============================

y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

# ============================
# 8. MÉTRIQUES
# ============================

print("\n=== Logistic Regression (baseline) ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_pred_proba))
print("Log-loss :", log_loss(y_test, y_pred_proba))


Train size : 5421
Test size  : 301

=== Logistic Regression (baseline) ===
Accuracy : 0.5481727574750831
AUC      : 0.5323868677905945
Log-loss : 0.6897606710117428


In [335]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

# ============================
# 1) y ALIGNÉ
# ============================
y = (
    df_features.loc[df_clean.index, "winner"]
    == df_features.loc[df_clean.index, "r_name"]
).astype(int)

# ============================
# 2) X NUMÉRIQUE
# ============================
cols_to_drop = [
    "date", "fight_id",
    "r_name", "b_name",
    "title_fight",
    "division", "r_stance", "b_stance",
]

X = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

# drop datetime cols (ex: r_dob/b_dob si jamais)
X = X.drop(columns=X.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns)

# drop rows with NaN (aligné)
valid_idx = X.dropna().index
X = X.loc[valid_idx]
y = y.loc[valid_idx]
df_tmp = df_clean.loc[valid_idx].copy()

assert X.select_dtypes(include="object").shape[1] == 0
assert X.isna().sum().sum() == 0

# ============================
# 3) SPLIT TEMPOREL (quantile recommandé)
# ============================
dates = pd.to_datetime(df_tmp["date"])
split_date = dates.quantile(0.8)  # 80% train, 20% test (temporel)

train_idx = dates < split_date
test_idx  = dates >= split_date

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print("Split date :", split_date.date())
print("Train size :", X_train.shape[0])
print("Test size  :", X_test.shape[0])

# ============================
# 4) XGBOOST (pas de scaling)
# ============================
pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = (neg / pos) if pos > 0 else 1.0

model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    reg_alpha=0.0,
    gamma=0.0,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42,
)

model.fit(X_train, y_train)

# ============================
# 5) METRICS + SEUIL OPTIMAL (accuracy)
# ============================
proba = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, proba)
ll  = log_loss(y_test, proba)

# seuil 0.5
pred_05 = (proba >= 0.5).astype(int)
acc_05 = accuracy_score(y_test, pred_05)

# seuil optimal sur le TEST (pour diagnostic) — pour un vrai pipeline, on le choisit sur validation train
thresholds = np.linspace(0.05, 0.95, 181)
accs = [accuracy_score(y_test, (proba >= t).astype(int)) for t in thresholds]
best_t = thresholds[int(np.argmax(accs))]
best_acc = float(np.max(accs))

print("\n=== XGBoost ===")
print(f"AUC      : {auc:.3f}")
print(f"Log-loss : {ll:.3f}")
print(f"Acc@0.50 : {acc_05:.3f}")
print(f"Best Acc : {best_acc:.3f}  (threshold={best_t:.2f})")

# ============================
# 6) (Option) FEATURES IMPORTANCE
# ============================
imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop 15 features:")
print(imp.head(15))


Split date : 2022-12-03
Train size : 4568
Test size  : 1154

=== XGBoost ===
AUC      : 0.491
Log-loss : 0.737
Acc@0.50 : 0.498
Best Acc : 0.548  (threshold=0.20)

Top 15 features:
r_weight             0.029025
b_losses             0.028770
b_splm               0.028202
b_draws              0.026161
r_fights_last_1y     0.026126
b_td_def             0.025962
r_age                0.025564
r_td_avg             0.025488
b_sub_avg            0.025347
r_sapm               0.025328
r_td_avg_acc         0.025249
b_td_avg             0.025236
r_n_fights_before    0.025115
b_td_avg_acc         0.025031
r_reach              0.024973
dtype: float32


# DIFF MORPHOLOGIQUE

In [304]:
df_diff = df_clean.copy()

df_diff["diff_height"] = df_diff["r_height"] - df_diff["b_height"]
df_diff["diff_weight"] = df_diff["r_weight"] - df_diff["b_weight"]
df_diff["diff_reach"]  = df_diff["r_reach"]  - df_diff["b_reach"]
df_diff["diff_age"]    = df_diff["r_age"]    - df_diff["b_age"]


# DIFF EXPERIENCE HISTORIQUE

In [305]:
df_diff["diff_wins"]   = df_diff["r_wins"]   - df_diff["b_wins"]
df_diff["diff_losses"] = df_diff["r_losses"] - df_diff["b_losses"]
df_diff["diff_draws"]  = df_diff["r_draws"]  - df_diff["b_draws"]

df_diff["diff_n_fights_before"] = (
    df_diff["r_n_fights_before"] - df_diff["b_n_fights_before"]
)


# DIFF KO SUB DEC

In [306]:
df_diff["diff_ko_before"]  = df_diff["r_ko_before"]  - df_diff["b_ko_before"]
df_diff["diff_sub_before"] = df_diff["r_sub_before"] - df_diff["b_sub_before"]

df_diff["diff_max_streak"] = (
    df_diff["r_max_win_streak_before"]
    - df_diff["b_max_win_streak_before"]
)


# DIFF ACTIVITE RECENTE 

In [307]:
df_diff["diff_days_since_last_fight"] = (
    df_diff["r_days_since_last_fight"]
    - df_diff["b_days_since_last_fight"]
)

df_diff["diff_fights_last_1y"] = (
    df_diff["r_fights_last_1y"]
    - df_diff["b_fights_last_1y"]
)


# DIFF STYLE 

In [308]:
df_diff["diff_splm"] = df_diff["r_splm"] - df_diff["b_splm"]
df_diff["diff_sapm"] = df_diff["r_sapm"] - df_diff["b_sapm"]

df_diff["diff_str_acc"] = df_diff["r_str_acc"] - df_diff["b_str_acc"]
df_diff["diff_str_def"] = df_diff["r_str_def"] - df_diff["b_str_def"]

df_diff["diff_td_avg"] = df_diff["r_td_avg"] - df_diff["b_td_avg"]
df_diff["diff_td_def"] = df_diff["r_td_def"] - df_diff["b_td_def"]

df_diff["diff_sub_avg"] = df_diff["r_sub_avg"] - df_diff["b_sub_avg"]


In [309]:
print(df_diff.shape)

(7907, 73)


# ENCODING DE LA DIVISION

In [310]:
import pandas as pd

df_diff = pd.concat(
    [
        df_diff,
        pd.get_dummies(df_diff['division'], prefix='div', drop_first=True)
    ],
    axis=1
)



# ENCODING DE LA STANCE

In [311]:
# One-hot commun
stance_oh = pd.get_dummies(
    pd.concat([df_diff['r_stance'], df_diff['b_stance']]),
    prefix='stance'
)

n = len(df_diff)

r_stance = stance_oh.iloc[:n].reset_index(drop=True).astype(int)
b_stance = stance_oh.iloc[n:].reset_index(drop=True).astype(int)

# Diff Red - Blue
for col in r_stance.columns:
    df_diff[col + '_diff'] = r_stance[col] - b_stance[col]
    




In [312]:
df_diff.columns

Index(['date', 'fight_id', 'division', 'title_fight', 'total_rounds', 'r_name',
       'r_wins', 'r_losses', 'r_draws', 'r_height', 'r_weight', 'r_reach',
       'r_stance', 'r_dob', 'r_splm', 'r_str_acc', 'r_sapm', 'r_str_def',
       'r_td_avg', 'r_td_avg_acc', 'r_td_def', 'r_sub_avg', 'b_name', 'b_wins',
       'b_losses', 'b_draws', 'b_height', 'b_weight', 'b_reach', 'b_stance',
       'b_dob', 'b_splm', 'b_str_acc', 'b_sapm', 'b_str_def', 'b_td_avg',
       'b_td_avg_acc', 'b_td_def', 'b_sub_avg', 'r_n_fights_before',
       'r_ko_before', 'r_sub_before', 'r_max_win_streak_before',
       'r_days_since_last_fight', 'r_fights_last_1y', 'b_n_fights_before',
       'b_ko_before', 'b_sub_before', 'b_max_win_streak_before',
       'b_days_since_last_fight', 'b_fights_last_1y', 'r_age', 'b_age',
       'diff_height', 'diff_weight', 'diff_reach', 'diff_age', 'diff_wins',
       'diff_losses', 'diff_draws', 'diff_n_fights_before', 'diff_ko_before',
       'diff_sub_before', 'diff_max_stre

# GARDER SEULEMENT LES VARIABLES DIFF , SUPPRIMER L'APPARTENANCE RED BLUE 

In [313]:
# Colonnes à garder explicitement
keep_cols = [
    'date',
    'title_fight',
    'total_rounds',
    'fight_id'
]

# Ajouter toutes les diff features
keep_cols += [c for c in df_diff.columns if c.startswith('diff_')]

# Ajouter division encodée
keep_cols += [c for c in df_diff.columns if c.startswith('div_')]

# Ajouter stance diff
keep_cols += [c for c in df_diff.columns if c.startswith('stance_')]

# Sous-dataset clean
df_diff_clean = df_diff[keep_cols].copy()

# shape
df_diff_clean.columns

Index(['date', 'title_fight', 'total_rounds', 'fight_id', 'diff_height',
       'diff_weight', 'diff_reach', 'diff_age', 'diff_wins', 'diff_losses',
       'diff_draws', 'diff_n_fights_before', 'diff_ko_before',
       'diff_sub_before', 'diff_max_streak', 'diff_days_since_last_fight',
       'diff_fights_last_1y', 'diff_splm', 'diff_sapm', 'diff_str_acc',
       'diff_str_def', 'diff_td_avg', 'diff_td_def', 'diff_sub_avg',
       'div_featherweight', 'div_flyweight', 'div_heavyweight',
       'div_light heavyweight', 'div_lightweight', 'div_middleweight',
       'div_welterweight', 'div_women's bantamweight', 'div_women's flyweight',
       'div_women's strawweight', 'stance_Open Stance_diff',
       'stance_Orthodox_diff', 'stance_Sideways_diff', 'stance_Southpaw_diff',
       'stance_Switch_diff'],
      dtype='object')

# RECUPERER LE WINNER DU COMBAT POUR CONSTRUIRE LA CIBLE

In [314]:
df_winner = (
    ufc[['fight_id', 'winner']]
    .drop_duplicates()
    .copy()
)

df_winner['fight_id'].is_unique


True

In [315]:
df_label = (
    df_diff[['fight_id', 'r_name']]
    .merge(df_winner, on='fight_id', how='left', validate='many_to_one')
)

df_label.isna().mean()



fight_id    0.000000
r_name      0.000000
winner      0.017832
dtype: float64

In [316]:
y = (df_label['r_name'] == df_label['winner']).astype(int)
y.value_counts(normalize=True)


1    0.625016
0    0.374984
Name: proportion, dtype: float64

In [317]:
X = df_diff_clean.sort_values('fight_id').reset_index(drop=True)
y = y.loc[X.index].reset_index(drop=True)


y.index = df_label['fight_id']
X.index = df_diff_clean['fight_id']

y = y.loc[X.index]


In [318]:
import numpy as np

np.random.seed(42)
swap_mask = np.random.rand(len(X)) < 0.5

diff_cols = [
    c for c in X.columns
    if c.startswith('diff_') or c.endswith('_diff')
]

# Inversion des features directionnelles
X.loc[swap_mask, diff_cols] *= -1

# Inversion du label
y.loc[swap_mask] = 1 - y.loc[swap_mask]


In [319]:
print("Swap ratio:", swap_mask.mean())
print("y mean:", y.mean())
print("diff means:", X[diff_cols].mean().abs().sort_values().head())

X.to_csv('X_final.csv', index=False)

Swap ratio: 0.5055014544074871
y mean: 0.49664853926900215
diff means: diff_sub_before            0.000000
stance_Sideways_diff       0.000126
stance_Switch_diff         0.000506
stance_Open Stance_diff    0.000759
diff_max_streak            0.001157
dtype: float64


In [320]:
X_model = X.drop(columns=['fight_id','date'])
X_model = X_model.drop(columns=[c for c in X_model.columns if c.startswith("div_")])
X_model.shape

(7907, 27)

# TRAIN TEST SPLIT TEMPORELLE

In [321]:
from sklearn.model_selection import TimeSeriesSplit

X_model = X_model.astype(float)

# y = label aligné

# Choix simple : 80% train / 20% test
split_date = X['date'].quantile(0.8)

train_idx = X['date'] <= split_date
test_idx  = X['date'] >  split_date

X_train = X_model.loc[train_idx]
X_test  = X_model.loc[test_idx]

y_train = y.loc[train_idx]
y_test  = y.loc[test_idx]

X_train.shape, X_test.shape
y_train.mean(), y_test.mean()

X_model.dtypes.value_counts()



float64    27
Name: count, dtype: int64

In [322]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss

pipe_logit = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        n_jobs=-1
    ))
])

# Fit
pipe_logit.fit(X_train, y_train)

# Prédictions
y_pred_proba = pipe_logit.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

# Metrics
print("AUC :", roc_auc_score(y_test, y_pred_proba))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("LogLoss :", log_loss(y_test, y_pred_proba))


AUC : 0.5591223501713856
Accuracy : 0.5404201145767027
LogLoss : 0.6883416289228775


In [323]:
import xgboost as xgb
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss

# =========================
# MODEL
# =========================
xgb_clf = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# =========================
# FIT (NO EARLY STOPPING)
# =========================
xgb_clf.fit(X_train, y_train)

# =========================
# EVALUATION
# =========================
y_pred_proba = xgb_clf.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print("AUC :", roc_auc_score(y_test, y_pred_proba))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("LogLoss :", log_loss(y_test, y_pred_proba))

# =========================
# FEATURE IMPORTANCE
# =========================
feat_imp = pd.Series(
    xgb_clf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(feat_imp.head(15))


AUC : 0.526833112277823
Accuracy : 0.5092297899427116
LogLoss : 0.7100872491265999
diff_n_fights_before          0.054889
stance_Orthodox_diff          0.046330
total_rounds                  0.045103
diff_age                      0.043689
diff_str_def                  0.043061
diff_fights_last_1y           0.042611
diff_wins                     0.042364
diff_splm                     0.042015
diff_sapm                     0.041880
diff_td_avg                   0.041849
diff_max_streak               0.041803
diff_days_since_last_fight    0.041709
diff_draws                    0.041575
diff_reach                    0.040938
stance_Switch_diff            0.040445
dtype: float32
